# Maia Lite 0.3.0 Chat — From-scratch pretraining (Colab)

**Research notebook · version 0.3.0 / pilot v1**

Goal: a **new** GPT-2-style causal model focused on **PT/EN/ES language, conversation, and logical reasoning**. This notebook **never loads 0.1/0.2 model weights**, and **does not perform SFT or CPT**. It trains a new tokenizer and initializes fresh weights.

**Important:** This is a *reproducible pilot/training pipeline*, not a ready-made large corpus. Supply curated, licensed data before production training. The optional tiny smoke-test dataset checks only that the pipeline runs; it cannot produce a useful language model.

Run cells in order. Mount Drive first, then upload corpus files to the `corpus/source` folder, configure the experiment, build tokenizer, tokenize, train, and evaluate. For a first run, use the pilot configuration. A 355M GPT-2 model can require an A100 80GB depending on context, batch and optimizer; an L4 may need a smaller pilot configuration.

## 1. Dependencies and Drive

In [ ]:
%pip -q install "transformers>=4.44,<5" "datasets>=3,<5" "tokenizers>=0.20,<0.23" "accelerate>=1,<2" "safetensors>=0.4" "sentencepiece>=0.2" "numpy>=1.26"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random, hashlib, math, time, shutil, sys
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast, Trainer, TrainingArguments, set_seed

ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
SOURCE = ROOT / 'corpus' / 'source'
WORK = ROOT / 'work'
TOKENIZER_DIR = ROOT / 'tokenizer'
MODEL_DIR = ROOT / 'checkpoints'
REPORTS = ROOT / 'reports'
for d in [SOURCE, WORK, TOKENIZER_DIR, MODEL_DIR, REPORTS]: d.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Configuration and data contract

Input files: `*.jsonl` (one object per line with `text`, `lang` = `pt`/`en`/`es`, `domain` = `language`/`dialogue`/`logic`/`educational`/`creative`), or `*.txt` stored under `corpus/source/<lang>/<domain>/...`. For `.txt`, **one paragraph per nonempty line**. JSONL is preferred because it preserves document boundaries and metadata.

Suggested **experimental** domain sampling: 30% language, 20% dialogue, 25% logic, 15% educational, 10% creative. Suggested languages: PT 40%, EN 35%, ES 25%. These are *sampling targets*, not verified optimal proportions. Actual composition is reported; this pilot **does not silently synthesize or oversample** missing domains. Prepare independent documents, record licenses/provenance, and remove test leakage before use.

In [ ]:
SEED = 42
set_seed(SEED)
# 'pilot' = ~50M parameters for pipeline validation; 'full' = GPT-2 medium-like ~355M
MODEL_SCALE = 'pilot'    # change to 'full' ONLY when corpus + compute are ready
VOCAB_SIZE = 40000
CONTEXT = 512 if MODEL_SCALE == 'pilot' else 1024
MAX_STEPS = 300 if MODEL_SCALE == 'pilot' else 20000  # deliberate user-set budget, NOT a convergence target
BATCH = 2 if MODEL_SCALE == 'pilot' else 1
GRAD_ACCUM = 8 if MODEL_SCALE == 'pilot' else 32
LR = 3e-4 if MODEL_SCALE == 'pilot' else 1.5e-4
WARMUP_STEPS = max(10, int(MAX_STEPS * .03))
SAVE_STEPS = 100 if MODEL_SCALE == 'pilot' else 500
LOG_STEPS = 10
VAL_PERCENT = 5
MAX_TOKENIZER_DOCS = 150000
MIN_DOC_CHARS = 30
ALLOW_SMOKE_TEST = False  # set True only for testing the notebook with artificial toy data
TARGET_LANG = {'pt': .40, 'en': .35, 'es': .25}
TARGET_DOMAIN = {'language': .30, 'dialogue': .20, 'logic': .25, 'educational': .15, 'creative': .10}
assert MODEL_SCALE in ('pilot','full')
CONFIG = dict(model_scale=MODEL_SCALE, vocab_size=VOCAB_SIZE, context=CONTEXT, seed=SEED,
              max_steps=MAX_STEPS, batch=BATCH, gradient_accumulation=GRAD_ACCUM, lr=LR)
print(json.dumps(CONFIG, indent=2))

## 3. Corpus audit, deterministic document split, deduplication

This stage rejects missing labels, duplicates (exact normalized text), and short records. The validation split is assigned **by document hash before tokenizer training**, preventing tokenizer fitting on validation documents. Exact deduplication is not near-deduplication; perform semantic/near-duplicate and provenance checks upstream. The script refuses to proceed with no corpus unless smoke-test mode is explicitly enabled.

In [ ]:
import unicodedata

def normalize(text):
    return unicodedata.normalize('NFC', str(text)).replace('\r\n', '\n').strip()

def iter_source_records():
    files = sorted(p for p in SOURCE.rglob('*') if p.is_file() and p.suffix.lower() in {'.jsonl','.txt'})
    if not files and ALLOW_SMOKE_TEST:
        # Synthetic smoke data is intentionally insufficient for language-model learning.
        examples = [
          ('pt','logic','Todos os daxes são mipos. Zaf é um dax. Portanto Zaf é um mipo.'),
          ('en','logic','All daxes are mipos. Zaf is a dax. Therefore Zaf is a mipo.'),
          ('es','logic','Todos los daxes son mipos. Zaf es un dax. Por tanto Zaf es un mipo.'),
          ('pt','dialogue','Usuário: Fale sobre a saudade. Assistente: A saudade é uma experiência de ausência e memória.'),
          ('en','language','Language conveys meaning through context, grammar, and discourse.'),
          ('es','creative','La memoria y la imaginación permiten contar historias sobre el pasado.'),
        ]
        for _ in range(300):
            for lang,domain,text in examples: yield dict(text=text+f' Exemplo {_}.',lang=lang,domain=domain,source='SMOKE_ONLY')
    elif not files:
        raise FileNotFoundError(f'No corpus in {SOURCE}. Upload data or explicitly enable smoke test.')
    for path in files:
        if path.suffix.lower() == '.jsonl':
            with path.open(encoding='utf-8') as f:
                for i,line in enumerate(f,1):
                    if not line.strip(): continue
                    try: item=json.loads(line)
                    except Exception as e: raise ValueError(f'{path}:{i}: {e}')
                    yield dict(text=item.get('text',''),lang=item.get('lang',''),domain=item.get('domain',''),source=str(path))
        else:
            try: lang,domain=path.relative_to(SOURCE).parts[:2]
            except ValueError: raise ValueError(f'Use source/<lang>/<domain>/<filename>.txt: {path}')
            with path.open(encoding='utf-8') as f:
                for line in f:
                    if line.strip(): yield dict(text=line,lang=lang,domain=domain,source=str(path))

train_path=WORK/'train_documents.jsonl'; val_path=WORK/'val_documents.jsonl'
seen=set(); counts=Counter(); rejected=Counter(); chars=Counter(); sample_records=[]
with train_path.open('w',encoding='utf-8') as tr, val_path.open('w',encoding='utf-8') as va:
    for rec in iter_source_records():
        txt=normalize(rec['text']); lang=rec['lang']; domain=rec['domain']
        if lang not in TARGET_LANG or domain not in TARGET_DOMAIN: rejected['invalid_metadata']+=1; continue
        if len(txt)<MIN_DOC_CHARS: rejected['short']+=1; continue
        digest=hashlib.sha256(txt.encode('utf-8')).hexdigest()
        if digest in seen: rejected['duplicate']+=1; continue
        seen.add(digest)
        is_val=int(digest[:12],16)%100 < VAL_PERCENT
        split='val' if is_val else 'train'
        record=dict(text=txt,lang=lang,domain=domain,sha256=digest)
        (va if is_val else tr).write(json.dumps(record,ensure_ascii=False)+'\n')
        counts[(split,lang,domain)]+=1; chars[(split,lang,domain)]+=len(txt)
assert sum(v for (s,*_),v in counts.items() if s=='train')>0, 'No training documents after filtering'
assert sum(v for (s,*_),v in counts.items() if s=='val')>0, 'No validation documents; add more independent records'
report={'counts':{str(k):v for k,v in counts.items()},'characters':{str(k):v for k,v in chars.items()},'rejected':dict(rejected)}
(REPORTS/'corpus_audit.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
print(json.dumps(report,indent=2,ensure_ascii=False)[:7000])

## 4. Train a new byte-level BPE tokenizer (training split only)

Special tokens are reserved from the outset. A tokenizer comparison (32k/40k/50k, token/character ratios by language) should precede a costly full run. Here we implement the selected 40k baseline.

In [ ]:
tok_input=WORK/'tokenizer_train.txt'
rng=random.Random(SEED)
# Reservoir sample for bounded RAM; training docs only.
reservoir=[]; seen_docs=0
with train_path.open(encoding='utf-8') as f:
    for line in f:
        rec=json.loads(line); seen_docs+=1
        if len(reservoir)<MAX_TOKENIZER_DOCS: reservoir.append(rec['text'])
        else:
            j=rng.randrange(seen_docs)
            if j<MAX_TOKENIZER_DOCS: reservoir[j]=rec['text']
with tok_input.open('w',encoding='utf-8') as f:
    for text in reservoir: f.write(text.replace('\n',' ')+'\n')
print('Tokenizer sample documents:',len(reservoir))
raw_tok=ByteLevelBPETokenizer()
SPECIAL=['<|pad|>','<|bos|>','<|eos|>','<|user|>','<|assistant|>','<|system|>']
raw_tok.train(files=[str(tok_input)], vocab_size=VOCAB_SIZE, min_frequency=2, special_tokens=SPECIAL)
raw_tok.save_model(str(TOKENIZER_DIR))
tokenizer=GPT2TokenizerFast(vocab_file=str(TOKENIZER_DIR/'vocab.json'),
                            merges_file=str(TOKENIZER_DIR/'merges.txt'),
                            bos_token='<|bos|>',eos_token='<|eos|>',pad_token='<|pad|>',
                            additional_special_tokens=['<|user|>','<|assistant|>','<|system|>'])
tokenizer.model_max_length=CONTEXT
tokenizer.save_pretrained(str(TOKENIZER_DIR))
print('Actual tokenizer size:',len(tokenizer),'EOS:',tokenizer.eos_token_id,'PAD:',tokenizer.pad_token_id)
assert len(tokenizer)<=VOCAB_SIZE
assert tokenizer.eos_token_id!=tokenizer.pad_token_id

## 5. Tokenize and pack by document, preserving EOS boundaries

Creates `train.bin` and `val.bin` as uint32 arrays. They are written atomically, so interrupted writes do not overwrite a completed version. Packed causal LM training concatenates documents separated by EOS; attention may cross document boundaries. For strict isolation, use document-aware attention masking in a later study.

In [ ]:
def encode_split(src,dst):
    tmp=dst.with_suffix('.bin.tmp')
    total=0; docs=0; token_counts=Counter()
    with src.open(encoding='utf-8') as fin, tmp.open('wb') as out:
        for line in fin:
            rec=json.loads(line)
            ids=tokenizer.encode(rec['text'],add_special_tokens=False)+[tokenizer.eos_token_id]
            np.asarray(ids,dtype=np.uint32).tofile(out)
            total+=len(ids); docs+=1; token_counts[(rec['lang'],rec['domain'])]+=len(ids)
    os.replace(tmp,dst)
    return {'documents':docs,'tokens':total,'tokens_by_group':{str(k):v for k,v in token_counts.items()}}
train_stats=encode_split(train_path,WORK/'train.bin')
val_stats=encode_split(val_path,WORK/'val.bin')
assert train_stats['tokens']>=CONTEXT+2 and val_stats['tokens']>=CONTEXT+2, 'Corpus too small for configured CONTEXT'
(REPORTS/'token_stats.json').write_text(json.dumps({'train':train_stats,'val':val_stats},indent=2))
print('Train:',train_stats,'\nValidation:',val_stats)
print('WARNING: this pipeline uses the observed corpus distribution, not the target weights. Review token_stats before full training.')

## 6. Initialize GPT-2 from scratch and train

Pilot configuration uses fewer layers; full configuration matches the historical GPT-2-medium geometry (`n_embd=1024, n_layer=24, n_head=16`) with a **new vocabulary**. Actual parameter count changes with vocabulary size. `max_steps` counts optimizer updates, not documents or epochs. The same packed token stream can be revisited repeatedly: inspect effective unique tokens and overfitting. Checkpoints contain optimizer and scheduler state; resume with `trainer.train(resume_from_checkpoint=...)`.

In [ ]:
from torch.utils.data import Dataset
class PackedDataset(Dataset):
    def __init__(self,path,context):
        self.path=path; self.context=context
        self.data=np.memmap(path,dtype=np.uint32,mode='r')
        self.n=max(0,(len(self.data)-1)//context)
    def __len__(self): return self.n
    def __getitem__(self,index):
        start=index*self.context
        ids=np.asarray(self.data[start:start+self.context+1],dtype=np.int64)
        return {'input_ids':torch.from_numpy(ids[:-1].copy()),'labels':torch.from_numpy(ids[1:].copy())}
train_ds=PackedDataset(WORK/'train.bin',CONTEXT)
val_ds=PackedDataset(WORK/'val.bin',CONTEXT)
assert len(train_ds)>0 and len(val_ds)>0
print('Nonoverlapping blocks:',len(train_ds),len(val_ds))

if MODEL_SCALE=='full': width,layers,heads=1024,24,16
else: width,layers,heads=512,6,8
model_config=GPT2Config(vocab_size=len(tokenizer),n_positions=CONTEXT,n_ctx=CONTEXT,
                        n_embd=width,n_layer=layers,n_head=heads,
                        bos_token_id=tokenizer.bos_token_id,eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.pad_token_id)
model=GPT2LMHeadModel(model_config)
model.gradient_checkpointing_enable()
print('Fresh model parameters:',sum(p.numel() for p in model.parameters()))
print('Effective token presentations budget (approx):',MAX_STEPS*BATCH*GRAD_ACCUM*CONTEXT)
print('Unique train tokens:',train_stats['tokens'])
assert MAX_STEPS*BATCH*GRAD_ACCUM*CONTEXT <= train_stats['tokens']*50 or ALLOW_SMOKE_TEST, 'High reuse of training tokens; revise corpus/budget' 

In [ ]:
# Save and check an experiment manifest so a restart does not silently mix configurations.
manifest_path=ROOT/'experiment_manifest.json'
manifest=dict(CONFIG,model_config=model_config.to_dict(),tokenizer_size=len(tokenizer),
              tokenizer_sha256=hashlib.sha256((TOKENIZER_DIR/'vocab.json').read_bytes()).hexdigest(),
              train_sha256=hashlib.sha256((WORK/'train.bin').read_bytes()).hexdigest(),
              val_sha256=hashlib.sha256((WORK/'val.bin').read_bytes()).hexdigest())
if manifest_path.exists():
    previous=json.loads(manifest_path.read_text())
    assert previous==manifest, 'Existing experiment manifest differs! Use a new ROOT or restore original settings.'
else: manifest_path.write_text(json.dumps(manifest,indent=2))
print('Manifest validated:',manifest_path)

In [ ]:
import inspect
args_kwargs=dict(output_dir=str(MODEL_DIR),overwrite_output_dir=False,
                 per_device_train_batch_size=BATCH,per_device_eval_batch_size=1,
                 gradient_accumulation_steps=GRAD_ACCUM,max_steps=MAX_STEPS,
                 learning_rate=LR,warmup_steps=WARMUP_STEPS,lr_scheduler_type='cosine',
                 weight_decay=.1,logging_steps=LOG_STEPS,save_steps=SAVE_STEPS,
                 save_strategy='steps',save_total_limit=3,report_to='none',
                 fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
                 bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
                 dataloader_num_workers=0,remove_unused_columns=False,
                 seed=SEED,save_safetensors=True)
# Evaluation strategy renamed in recent transformers releases.
arg_params=inspect.signature(TrainingArguments).parameters
if 'eval_strategy' in arg_params: args_kwargs.update(eval_strategy='steps',eval_steps=SAVE_STEPS)
else: args_kwargs.update(evaluation_strategy='steps',eval_steps=SAVE_STEPS)
args=TrainingArguments(**args_kwargs)
trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds)
checkpoints=sorted([p for p in MODEL_DIR.glob('checkpoint-*') if p.is_dir() and (p/'trainer_state.json').exists()],
                   key=lambda p:int(p.name.split('-')[-1]))
resume=str(checkpoints[-1]) if checkpoints else None
print('Resuming from:',resume or 'fresh initialization')
result=trainer.train(resume_from_checkpoint=resume)
print(result.metrics)
trainer.save_model(str(ROOT/'final_model'))
tokenizer.save_pretrained(str(ROOT/'final_model'))
trainer.save_state()

## 7. Validation and regression probes

Qualitative generation at this stage is **not an instruction-following benchmark**: a base LM may continue the prompt rather than answer it. Compare versions with identical prompts and decoding, and save raw outputs. These probes are intended to catch regressions like `Fale sobre a saudade` vs `O que é a saudade?`. A separate structured benchmark with held-out logical templates, human ratings and repeated seeds is required for scientific conclusions.

In [ ]:
metrics=trainer.evaluate()
print('Held-out metrics:',metrics)
if 'eval_loss' in metrics: print('Perplexity (exp loss):',math.exp(min(30,metrics['eval_loss'])))
probes=[
 ('pt','O que é a saudade?'),('pt','Fale sobre a saudade.'),
 ('pt','Explique a diferença entre saudade e nostalgia.'),
 ('en','What is nostalgia?'),('en','Talk about nostalgia.'),
 ('es','¿Qué es la nostalgia?'),('es','Habla sobre la nostalgia.'),
 ('pt','Todos os daxes são mipos. Zaf é um dax. Zaf é um mipo?'),
 ('en','All daxes are mipos. Zaf is a dax. Is Zaf a mipo?'),
 ('es','Todos los daxes son mipos. Zaf es un dax. ¿Zaf es un mipo?'),
 ('pt','Se chove, a rua fica molhada. A rua está molhada. Necessariamente choveu?'),
]
model.eval(); device=next(model.parameters()).device
outputs=[]
for lang,prompt in probes:
    inputs=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=CONTEXT-128).to(device)
    with torch.no_grad():
        generated=model.generate(**inputs,max_new_tokens=100,do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    completion=tokenizer.decode(generated[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True)
    outputs.append(dict(lang=lang,prompt=prompt,completion=completion))
    print(f'[{lang}] {prompt}\n  => {completion[:350]}\n')
(REPORTS/'generation_probes.json').write_text(json.dumps(outputs,indent=2,ensure_ascii=False))
(REPORTS/'eval_metrics.json').write_text(json.dumps(metrics,indent=2))
print('Saved to:',REPORTS)

## 8. Before scaling to the full ~355M run

1. Replace smoke data with a licensed, provenance-tracked PT/EN/ES corpus. Ensure **independent held-out sources**, not just a random split of near-duplicate paragraphs.
2. Audit **actual token shares** by language and domain. Current pipeline reports shares but **does not enforce target mixture**; implement document-aware weighted sampling if targets are essential.
3. Compare tokenizers on held-out token/character ratio per language. Changing tokenizer means training a **new model**.
4. Benchmark `pilot` against a matched general-corpus control with equal token budget; then run `full` only if supported by evidence.
5. Keep validation probes fixed and test logical generalization using **novel entities and unseen inference templates**. Distinguish surface completion from correctness.
6. Preserve optimizer, scheduler, model and tokenizer with checkpoints; avoid restarting from model weights alone.
7. The `full` configuration uses a fixed-size ~355M architecture but new vocabulary changes exact parameter count; confirm GPU memory and throughput first.
8. **Do not run SFT automatically.** Decide later based on base-model capabilities and regression tests.

**Research note:** This notebook implements a *single experimental arm*. A publishable comparison requires controlled baselines, multiple random seeds, corpus licensing/contamination audits, evaluation protocols and uncertainty estimates.